# Part 1 Recovery: restore split checkpoints to temporary HF folders

This notebook tries to rebuild temporary HuggingFace inference folders for the split-format models:

- `bert_adversarial`
- `bert_attr_reg`

Expected source layout from the original training notebooks:

- `bert/` backbone folder saved with `save_pretrained()`
- `classifier.pt`
- tokenizer files in the model root
- `label_encoder.joblib`
- optional `city_encoder.joblib`

If enough pieces exist, the notebook creates a temporary HF-style folder under `results/restored_models/` and verifies that `AutoModelForSequenceClassification.from_pretrained()` can load it.


In [ ]:
import json
import shutil
from pathlib import Path

import pandas as pd
import torch
from transformers import AutoConfig, AutoModelForSequenceClassification, AutoTokenizer

CURRENT = Path.cwd().resolve()
WORKSPACE = CURRENT if (CURRENT / 'city-swap-eval').exists() else CURRENT.parent
CITY_SWAP_REPO = WORKSPACE / 'city-swap-eval'
OUTPUT_DIR = CITY_SWAP_REPO / 'results' / 'restored_models'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Workspace :', WORKSPACE)
print('Output dir:', OUTPUT_DIR)


In [ ]:
MODEL_SPECS = {
    'adversarial': {
        'folder_name': 'bert_adversarial',
    },
    'attr_reg': {
        'folder_name': 'bert_attr_reg',
    },
}

SEARCH_ROOTS = [
    WORKSPACE,
    WORKSPACE / 'resume-screening',
    WORKSPACE / 'city-swap-eval',
    WORKSPACE / '.tmp-repo-sync',
]
SEARCH_ROOTS = [p for p in SEARCH_ROOTS if p.exists()]
SEARCH_ROOTS


In [ ]:
def safe_rel(path: Path) -> str:
    try:
        return str(path.relative_to(WORKSPACE))
    except Exception:
        return str(path)


def list_matches(name: str):
    matches = []
    for root in SEARCH_ROOTS:
        try:
            matches.extend(sorted(root.rglob(name)))
        except Exception:
            pass
    unique = []
    seen = set()
    for path in matches:
        key = str(path.resolve())
        if key not in seen:
            unique.append(path)
            seen.add(key)
    return unique


def first_existing(paths):
    for path in paths:
        path = Path(path)
        if path.exists():
            return path
    return None


def load_state_dict_any(path: Path):
    obj = torch.load(path, map_location='cpu')
    if isinstance(obj, dict):
        for key in ['state_dict', 'model_state_dict', 'classifier_state_dict']:
            if key in obj and isinstance(obj[key], dict):
                return obj[key]
        return obj
    if hasattr(obj, 'state_dict'):
        return obj.state_dict()
    raise TypeError(f'Unsupported checkpoint object type: {type(obj)}')


def infer_num_labels(label_encoder_path: Path, training_config_path: Path = None):
    if label_encoder_path and label_encoder_path.exists():
        import joblib
        le = joblib.load(label_encoder_path)
        if hasattr(le, 'classes_'):
            return len(le.classes_)
    if training_config_path and training_config_path.exists():
        cfg = json.loads(training_config_path.read_text())
        if 'num_classes' in cfg:
            return int(cfg['num_classes'])
    return 9


def find_candidate_folder(folder_name: str):
    matches = list_matches(folder_name)
    return matches[0] if matches else None


def collect_source_paths(folder_name: str):
    folder = find_candidate_folder(folder_name)
    if folder is None:
        return None
    folder = Path(folder)
    return {
        'folder': folder,
        'bert_dir': first_existing([folder / 'bert']),
        'classifier_pt': first_existing([folder / 'classifier.pt']),
        'model_pt': first_existing([folder / 'model.pt']),
        'label_encoder': first_existing([folder / 'label_encoder.joblib']),
        'city_encoder': first_existing([folder / 'city_encoder.joblib']),
        'training_config': first_existing([folder / 'training_config.json']),
        'tokenizer_json': first_existing([folder / 'tokenizer.json']),
        'tokenizer_config': first_existing([folder / 'tokenizer_config.json']),
        'special_tokens_map': first_existing([folder / 'special_tokens_map.json']),
        'vocab_txt': first_existing([folder / 'vocab.txt']),
    }


In [ ]:
def remap_classifier_state(raw_state: dict, target_state: dict):
    remapped = {}
    for key, value in raw_state.items():
        norm_key = key.replace('module.', '')
        if norm_key.startswith('classifier.'):
            norm_key = norm_key[len('classifier.'): ]
        if norm_key in target_state and getattr(value, 'shape', None) == target_state[norm_key].shape:
            remapped[norm_key] = value
    return remapped


def build_restored_folder(model_name: str, src: dict):
    restore_dir = OUTPUT_DIR / model_name
    if restore_dir.exists():
        shutil.rmtree(restore_dir)
    restore_dir.mkdir(parents=True, exist_ok=True)

    if src['bert_dir'] is None or not src['bert_dir'].exists():
        raise FileNotFoundError('Missing bert/ backbone folder')
    if src['classifier_pt'] is None or not src['classifier_pt'].exists():
        raise FileNotFoundError('Missing classifier.pt')
    if src['label_encoder'] is None or not src['label_encoder'].exists():
        raise FileNotFoundError('Missing label_encoder.joblib')

    config = AutoConfig.from_pretrained(str(src['bert_dir']))
    config.num_labels = infer_num_labels(src['label_encoder'], src['training_config'])
    config.architectures = ['BertForSequenceClassification']

    model = AutoModelForSequenceClassification.from_pretrained(
        str(src['bert_dir']),
        config=config,
        ignore_mismatched_sizes=True,
    )

    raw_classifier_state = load_state_dict_any(src['classifier_pt'])
    target_state = model.classifier.state_dict()
    remapped = remap_classifier_state(raw_classifier_state, target_state)
    if set(remapped.keys()) != set(target_state.keys()):
        raise RuntimeError(
            f'Classifier state mismatch. Restored keys: {sorted(remapped.keys())}; expected: {sorted(target_state.keys())}'
        )
    model.classifier.load_state_dict(remapped, strict=True)

    model.save_pretrained(str(restore_dir), safe_serialization=True)

    for extra in ['label_encoder', 'city_encoder', 'training_config', 'tokenizer_json', 'tokenizer_config', 'special_tokens_map', 'vocab_txt']:
        src_path = src.get(extra)
        if src_path is not None and Path(src_path).exists():
            shutil.copy2(src_path, restore_dir / Path(src_path).name)

    tokenizer_source = src['folder'] if src['tokenizer_json'] is not None else src['bert_dir']
    tokenizer = AutoTokenizer.from_pretrained(str(tokenizer_source))
    tokenizer.save_pretrained(str(restore_dir))

    return restore_dir


In [ ]:
def verify_restored_folder(model_name: str, restore_dir: Path):
    info = {
        'model': model_name,
        'restore_dir': safe_rel(restore_dir),
        'load_ok': False,
        'tokenizer_ok': False,
        'num_labels': None,
        'classifier_shape': '',
        'error': '',
    }
    try:
        tokenizer = AutoTokenizer.from_pretrained(str(restore_dir))
        model = AutoModelForSequenceClassification.from_pretrained(str(restore_dir))
        info['tokenizer_ok'] = True
        info['load_ok'] = True
        info['num_labels'] = int(model.config.num_labels)
        weight = model.classifier.weight
        info['classifier_shape'] = 'x'.join(str(x) for x in weight.shape)
        _ = tokenizer('test resume text', return_tensors='pt')
    except Exception as exc:
        info['error'] = f'{type(exc).__name__}: {exc}'
    return info


In [ ]:
rows = []
for model_name, spec in MODEL_SPECS.items():
    src = collect_source_paths(spec['folder_name'])
    row = {
        'model': model_name,
        'folder_name': spec['folder_name'],
        'source_folder': safe_rel(src['folder']) if src else '',
        'bert_dir': safe_rel(src['bert_dir']) if src and src['bert_dir'] else '',
        'classifier_pt': safe_rel(src['classifier_pt']) if src and src['classifier_pt'] else '',
        'label_encoder': safe_rel(src['label_encoder']) if src and src['label_encoder'] else '',
        'training_config': safe_rel(src['training_config']) if src and src['training_config'] else '',
        'restored': False,
        'restore_dir': '',
        'verified_load': False,
        'error': '',
    }
    if src is None:
        row['error'] = 'Source folder not found in workspace'
        rows.append(row)
        continue
    try:
        restore_dir = build_restored_folder(model_name, src)
        verify = verify_restored_folder(model_name, restore_dir)
        row['restored'] = True
        row['restore_dir'] = verify['restore_dir']
        row['verified_load'] = verify['load_ok']
        row['error'] = verify['error']
    except Exception as exc:
        row['error'] = f'{type(exc).__name__}: {exc}'
    rows.append(row)

restore_df = pd.DataFrame(rows)
restore_df.to_csv(OUTPUT_DIR / 'restore_summary.csv', index=False)
restore_df


In [ ]:
verification_rows = []
for _, row in restore_df.iterrows():
    if not row['restored']:
        verification_rows.append({
            'model': row['model'],
            'restore_dir': row['restore_dir'],
            'load_ok': False,
            'tokenizer_ok': False,
            'num_labels': None,
            'classifier_shape': '',
            'error': row['error'],
        })
        continue
    verification_rows.append(verify_restored_folder(row['model'], OUTPUT_DIR / row['model']))

verification_df = pd.DataFrame(verification_rows)
verification_df.to_csv(OUTPUT_DIR / 'restore_verification.csv', index=False)
verification_df


In [ ]:
final_df = restore_df.merge(
    verification_df[['model', 'load_ok', 'tokenizer_ok', 'num_labels', 'classifier_shape', 'error']],
    on='model',
    how='left',
    suffixes=('', '_verify'),
)
final_df.to_csv(OUTPUT_DIR / 'restore_final_report.csv', index=False)
final_df
